In [ ]:
import sys
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os

# Step 1
# download data to Colab
for ens in np.arange(1,11):
    os.system('wget -O HW5_ens'+str(ens).zfill(2)+'.nc https://zenodo.org/record/7384713/files/HW5_ens'+str(ens).zfill(2)+'.nc?download=1')


In [ ]:
!pip install netCDF4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 39.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.3 MB/s eta 0:00:00


In [ ]:
# the nc file has 4 dimensions: initial time (time), lead time (step), latitude, longitude
# ens01 indicates the first ensemble member and ens02 is the second ensemble member
import netCDF4


In [ ]:
gh = np.zeros((10,8,47,121,240)) # number of ensemble, initialization time, forecast lead, lat, lon
for i in range(1,11):
    fp     = '/content/HW5_ens'+str(i).zfill(2)+'.nc'
    nc     = netCDF4.Dataset(fp)
    gh[i-1,] = nc['gh'][:]
y = nc['latitude'][:]
x = nc['longitude'][:]
lon,lat = np.meshgrid(x,y)

In [ ]:
gh_weighted = gh * np.cos(lat/180*np.pi)**0.5
gh_weighted = np.reshape(gh_weighted,[int(np.size(gh_weighted)/(np.size(y)*np.size(x))),int(np.size(y)*np.size(x))])

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
modes = 100
pca = PCA(n_components=modes)
pca.fit(gh_weighted)

PCA(n_components=100)

In [ ]:
# Step 2
# dimension reduction
# derive EOF pattern (spatial structure) and principal components (time series)
EOF     = np.reshape(pca.components_,[modes,np.size(y),np.size(x)])
pcs     = np.dot(gh_weighted,pca.components_.transpose())
pcs_std = np.reshape(np.std(pcs,axis=0),[100,1])
pcs     = pcs/np.std(pcs_std,axis=0)
pcs     = np.reshape(pcs,[10,8,47,modes])


In [ ]:
# Step 3
# calculate deviation from ensemble mean
gh_prime  = pcs - pcs.mean(axis=0)